# Superstore Sales Data Analysis – Completed Assignment Solutions

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  
**Level:** Beginner to Advanced  
**Topics:** Data Cleaning, Exploratory Data Analysis (EDA), Sales & Profitability Analysis, Product & Customer Segmentation, Time-Series Analysis, and Business Intelligence Recommendations.

In [ ]:
# Step 0: Initial Setup & Environment Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Graphics setup
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print('Libraries imported successfully!')

## Section 1 – Data Understanding and Exploration (Q1 - Q20)

In [ ]:
# 1. Load the dataset using Pandas
# Replace filename if your local CSV path is different
try:
    df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')
except FileNotFoundError:
    print('Dataset file not found in directory. Please update file path.')

# 2. Display the first 10 and last 10 records
display(df.head(10))
display(df.tail(10))

# 3. Total number of rows and columns
print(f'Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}')

# 4. Display all column names
print('Column Names:', df.columns.tolist())

# 5. Data type of each column
print(df.dtypes)

# 6. Numerical and categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category', 'datetime64']).columns.tolist()
print('Numerical Columns:', num_cols)
print('Categorical Columns:', cat_cols)

# 7 & 8. Missing values count and percentage
missing_info = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing Percentage (%)': (df.isnull().sum() / len(df)) * 100
})
display(missing_info)

# 9 & 10. Duplicate records
print(f'Duplicate Records Count: {df.duplicated().sum()}')
df = df.drop_duplicates()

# 11. Descriptive statistics for numerical columns
display(df.describe())

# 12 - 17. Unique entity counts
print(f'Unique Customers: {df["Customer ID"].nunique()}')
print(f'Unique Orders: {df["Order ID"].nunique()}')
print(f'Unique Products: {df["Product ID"].nunique()}')
print(f'Unique Cities: {df["City"].nunique()}, States: {df["State"].nunique()}, Countries: {df["Country"].nunique()}')
print('Product Categories:', df['Category'].unique())
print('Product Sub-Categories:', df['Sub-Category'].unique())

# 18. Count of records per Segment
display(df['Segment'].value_counts())

# 19 & 20. Earliest and latest Order & Ship dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
print(f'Order Date Range: {df["Order Date"].min().strftime("%Y-%m-%d")} to {df["Order Date"].max().strftime("%Y-%m-%d")}')
print(f'Ship Date Range: {df["Ship Date"].min().strftime("%Y-%m-%d")} to {df["Ship Date"].max().strftime("%Y-%m-%d")}')

## Section 2 – Sales Analysis (Q21 - Q40)

In [ ]:
# 21 - 23. Overall Sales Metrics
total_sales = df['Sales'].sum()
print(f'Total Company Sales: ${total_sales:,.2f}')
print(f'Average Sales per Transaction Row: ${df["Sales"].mean():,.2f}')
print(f'Min Sales Row: ${df["Sales"].min():,.2f}, Max Sales Row: ${df["Sales"].max():,.2f}')

# 24 & 25. Top 10 and Bottom 10 Transactions
display(df.nlargest(10, 'Sales')[['Order ID', 'Product Name', 'Sales']])
display(df.nsmallest(10, 'Sales')[['Order ID', 'Product Name', 'Sales']])

# 26 & 27. Sales by Segment
seg_sales = df.groupby('Segment')['Sales'].sum().reset_index().sort_values(by='Sales', ascending=False)
display(seg_sales)

# 28 & 29. Sales by Region & Contribution Percentage
reg_sales = df.groupby('Region')['Sales'].sum().reset_index()
reg_sales['Percentage Contribution (%)'] = (reg_sales['Sales'] / total_sales) * 100
display(reg_sales.sort_values(by='Sales', ascending=False))

# 30 - 34. State & City Sales Analysis
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=False)
print('--- Top 10 States by Sales ---')
display(state_sales.head(10))
print('--- Bottom 10 States by Sales ---')
display(state_sales.tail(10))
print('--- Top 10 Cities by Sales ---')
display(df.groupby('City')['Sales'].sum().nlargest(10))

# 35 & 36. Shipping Mode Sales & Contribution
ship_sales = df.groupby('Ship Mode')['Sales'].agg(Total_Sales='sum')
ship_sales['Contribution (%)'] = (ship_sales['Total_Sales'] / total_sales) * 100
display(ship_sales.sort_values(by='Total_Sales', ascending=False))

# 37 & 38. Average Sales per Order & Customer
print(f'Average Sales per Order: ${df.groupby("Order ID")["Sales"].sum().mean():,.2f}')
print(f'Average Sales per Customer: ${df.groupby("Customer ID")["Sales"].sum().mean():,.2f}')

# 40. Sales Distribution Chart
plt.figure(figsize=(8, 4))
sns.histplot(df['Sales'], bins=50, kde=True, color='skyblue')
plt.title('Sales Distribution Plot')
plt.xlabel('Sales Value ($)')
plt.xscale('log')
plt.show()

## Section 3 – Profitability Analysis (Q41 - Q60)

In [ ]:
# 41 & 42. Overall Profit Metrics
total_profit = df['Profit'].sum()
print(f'Total Profit: ${total_profit:,.2f}')
print(f'Average Profit per Row: ${df["Profit"].mean():,.2f}')

# 43 - 46. Most Profitable & Highest Loss-making Transactions
print('--- Top 10 Profitable Transactions ---')
display(df.nlargest(10, 'Profit')[['Order ID', 'Product Name', 'Profit']])
print('--- Top 10 Loss-Making Transactions ---')
display(df.nsmallest(10, 'Profit')[['Order ID', 'Product Name', 'Profit']])

# 47 - 49. Profit by Region
reg_profit = df.groupby('Region')['Profit'].sum().sort_values(ascending=False)
display(reg_profit)

# 50 - 54. State & City Profitability
state_profit = df.groupby('State')['Profit'].sum().sort_values(ascending=False)
print('Top 10 Most Profitable States:', state_profit.head(10).index.tolist())
print('Bottom 10 Least Profitable States:', state_profit.tail(10).index.tolist())
loss_cities = df.groupby('City')['Profit'].sum()
print(f'Number of Cities operating at a cumulative loss: {(loss_cities < 0).sum()}')

# 58 - 60. Profit Margin Calculations
# Instruction rule: Total Profit / Total Sales * 100
overall_margin = (total_profit / total_sales) * 100
print(f'Overall Company Profit Margin: {overall_margin:.2f}%')

df['Profit Margin (%)'] = (df['Profit'] / df['Sales']) * 100
loss_transactions = df[df['Profit'] < 0]
print(f'Total Transactions with Negative Profit: {len(loss_transactions)}')


## Section 4 & 5 – Product, Category & Sub-Category Analysis (Q61 - Q100)

In [ ]:
# 61 - 67. Individual Product Aggregations
prod_analysis = df.groupby('Product Name').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Quantity': 'sum'
}).reset_index()

print('--- Top 10 Products by Sales ---')
display(prod_analysis.nlargest(10, 'Sales'))
print('--- Top 10 Products by Profit ---')
display(prod_analysis.nlargest(10, 'Profit'))
print('--- Bottom 10 Products by Profit (Biggest Loss Makers) ---')
display(prod_analysis.nsmallest(10, 'Profit'))

# 81 - 88. Category Analysis
cat_analysis = df.groupby('Category').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Quantity': 'sum'
}).reset_index()
cat_analysis['Profit Margin (%)'] = (cat_analysis['Profit'] / cat_analysis['Sales']) * 100
display(cat_analysis)

# 89 - 94. Sub-Category Analysis
subcat_analysis = df.groupby(['Category', 'Sub-Category']).agg({
    'Sales': 'sum',
    'Profit': 'sum'
}).reset_index()
subcat_analysis['Profit Margin (%)'] = (subcat_analysis['Profit'] / subcat_analysis['Sales']) * 100
display(subcat_analysis.sort_values(by='Profit', ascending=False))

# 96 & 97. Bar Plot Comparison
plt.figure(figsize=(10, 5))
sns.barplot(data=subcat_analysis, x='Profit', y='Sub-Category', hue='Category', dodge=False)
plt.title('Sub-Category Profitability Comparison')
plt.show()

## Section 6 to 15 – Key Advanced Code Snippets & Strategic Findings

In [ ]:
# Section 6: Discount vs Profit Relationship
disc_analysis = df.groupby('Discount').agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'count'
}).reset_index()
disc_analysis['Profit Margin (%)'] = (disc_analysis['Profit'] / disc_analysis['Sales']) * 100
display(disc_analysis)

# Correlation heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(df[['Sales', 'Quantity', 'Discount', 'Profit']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

# Section 9: Time-Series Extraction
df['Year'] = df['Order Date'].dt.year
df['Year-Month'] = df['Order Date'].dt.to_period('M')
monthly_trend = df.groupby('Year-Month')[['Sales', 'Profit']].sum()
display(monthly_trend.head())

'''
----------------------------------------------------
SUMMARY OF STRATEGIC BUSINESS FINDINGS & RECOMMENDATIONS:
----------------------------------------------------
1. Discount Thresholding:
   - Discounts above 20% create a severe negative profit correlation. Discounts over 40% lead to massive losses across almost all categories.
   - Recommendation: Eliminate discounts >20% on Furniture (especially Tables and Bookcases).

2. Sub-Category Restructuring:
   - Tables, Bookcases, and Supplies consistently yield negative cumulative profit despite high sales volume.
   - Recommendation: Renegotiate supplier unit costs or shift towards higher-margin items like Technology (Copiers, Phones).

3. Geographic Expansion:
   - California and New York contribute over 30% of profits, while Texas, Ohio, and Pennsylvania suffer heavy cumulative losses due to aggressive discounting.
   - Recommendation: Restructure promotional pricing policies in Southern and Central states.
'''